In [ ]:
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')
sp.init_printing()
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)

Setup complete. SymPy 1.14.0 | NumPy 2.0.2


In [ ]:
def f(x):
    return x**2

h = 1e-8
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 8)')

Numerical f'(3): 6.0  (exact = 8)


In [ ]:
expr = x**2
deriv = sp.diff(expr, x)
print('d/dx (x**2) =', deriv)
print('Symbolic f\'(3) =', deriv.subs(x, 3))

d/dx (x**2) = 2*x
Symbolic f'(3) = 6


In [ ]:
def g(x):
    return x**3 + 2*x
h = 1e-8
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 8)')

print('d/dx (x**2) =', deriv)

print('Symbolic f\'(2) =', deriv.subs(x, 2))


Numerical f'(3): 6.0  (exact = 8)
d/dx (x**2) = 2*x
Symbolic f'(2) = 4


In [ ]:
f2 = x**2 + 3*x*y + y**2

print('df/dx =', sp.diff(f2, x))
print('df/dy =', sp.diff(f2, y))

df/dx = 2*x + 3*y
df/dy = 3*x + 2*y


In [ ]:
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)

grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)

grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


In [ ]:
h2 = x**2 * y + sp.sin(y)

print('dh/dx =', sp.diff(h2, x))
print('dh/dy =', sp.diff(h2, y))

grad = [sp.diff(h2, v) for v in (x, y)]
print('grad h =', grad)

grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad h at (1, 2) =', grad_at)

dh/dx = 2*x*y
dh/dy = x**2 + cos(y)
grad h = [2*x*y, x**2 + cos(y)]
grad h at (1, 2) = [4, cos(2) + 1]


In [ ]:
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)

By hand : 2*x*cos(x**2)
By SymPy: 2*x*cos(x**2)
Match?   True


In [ ]:
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))

d/dx (3x+1)^4 = 12*(3*x + 1)**3


In [ ]:
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1
h  = np.maximum(0, z1)
y_hat = h @ W2
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))

Initial loss: 1.6936


In [ ]:
dy   = 2 * (y_hat - Y) / Y.size
dW2  = h.T @ dy
dh   = dy @ W2.T
dz1  = dh * (z1 > 0)
dW1  = X.T @ dz1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')

dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)


In [ ]:
lr = 0.1
W1 -= lr * dW1
W2 -= lr * dW2


h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

Loss before: 1.6936
Loss after : 1.6524 -> should be lower


In [ ]:
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1
Wb = np.random.randn(8, 1) * 0.1

lr = 0.1
W1 -= lr * dW1
W2 -= lr * dW2


h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

Loss before: 1.6936
Loss after : 1.6017 -> should be lower


In [20]:
import numpy as np

Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output

lr = 0.05

# 1. Forward pass
z1 = Xb @ Wa
h = np.maximum(0, z1)                # ReLU
y_hat = h @ Wb

loss_before = np.mean((y_hat - Yb) ** 2)

# 2. Backward pass
# d/d(y_hat) MSE = 2 * (y_hat - Yb) / number of elements
dy = 2 * (y_hat - Yb) / Yb.size

dWb = h.T @ dy
dh = dy @ Wb.T

# ReLU derivative
dz1 = dh * (z1 > 0)

dWa = Xb.T @ dz1

# 3. One gradient-descent step
Wa -= lr * dWa
Wb -= lr * dWb

# Forward pass again to get loss after update
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb

loss_after = np.mean((y_hat - Yb) ** 2)

print("Loss before:", loss_before)
print("Loss after: ", loss_after)

Loss before: 1.1706051565963387
Loss after:  1.161431476425855


In [21]:
f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)

Hessian of f:
⎡2  3⎤
⎢    ⎥
⎣3  2⎦


In [22]:
xv = 0.0
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)
    xv = xv - lr * grad
print('Converged x:', round(xv, 3), ' (true minimum = 4)')

Converged x: 3.998  (true minimum = 4)


In [25]:
# 1. Hessian of x**4 + y**2
# f(x, y) = x^4 + y^2
# Hessian = [[d²f/dx², d²f/dxdy],
#            [d²f/dydx, d²f/dy²]]

import sympy as sp

x, y = sp.symbols('x y')
f = x**4 + y**2

H = sp.hessian(f, (x, y))
print("Hessian:")
print(H)



xv = 0.0
lr = 0.1

for step in range(20):
    grad = 2 * (xv - 7)
    xv = xv - lr * grad

print('Final x:', xv)

Hessian:
Matrix([[12*x**2, 0], [0, 2]])
Final x: 6.91929549467752


PANDAS

In [24]:
import pandas as pd
import numpy as np
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'])
print("\nSeries:")
print(s)


data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Age': [25, 30, 35, 40],
    'Salary': [50000, 60000, 70000, 80000]
}
df = pd.DataFrame(data)
print("\nDataFrame:")
print(df)


Series:
a    10
b    20
c    30
d    40
dtype: int64

DataFrame:
      Name  Age  Salary
0    Alice   25   50000
1      Bob   30   60000
2  Charlie   35   70000
3    David   40   80000
